# PINN Project 1: 1D Heat Equation (Steady-State)
## Physics-Informed Neural Network for Beginners

### What is a PINN?
A **Physics-Informed Neural Network (PINN)** is a neural network that is trained not only
on data, but also on the **governing physics equations** (PDEs). The physics is embedded
directly into the loss function, so the network learns to satisfy the physical laws.

### Problem Description
We solve the **1D steady-state heat equation**:

$$\frac{d^2 u}{dx^2} = -\sin(\pi x), \quad x \in [0, 1]$$

with boundary conditions:
- $u(0) = 0$
- $u(1) = 0$

The **exact solution** is: $u(x) = \frac{\sin(\pi x)}{\pi^2}$

### Why this problem?
- Simplest possible PDE (1D, steady-state, linear)
- Has an exact solution so we can verify our PINN
- Only one independent variable (x), easy to visualize


In [ ]:
# ============================================================
# STEP 1: Import Libraries
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Step 2: Define the Neural Network

The network takes **x** as input and outputs **u(x)** (temperature).

Key concepts:
- We use a simple feedforward network (MLP)
- `tanh` activation works well for PINNs because it is smooth and differentiable
- We need smooth outputs because the loss function involves **derivatives** of u


In [ ]:
# ============================================================
# STEP 2: Define the Neural Network Architecture
# ============================================================

class PINN(nn.Module):
    """
    Physics-Informed Neural Network for the 1D heat equation.
    
    Architecture: x -> [Linear -> Tanh] x N_layers -> Linear -> u(x)
    
    Why tanh?
    - tanh is infinitely differentiable (C∞)
    - PINNs require computing derivatives of the output w.r.t. input
    - ReLU has discontinuous 2nd derivative, which causes problems for PDEs
    """
    
    def __init__(self, n_hidden=3, n_neurons=32):
        """
        Args:
            n_hidden (int): Number of hidden layers
            n_neurons (int): Number of neurons per hidden layer
        """
        super().__init__()
        
        # Build the network layer by layer
        layers = []
        
        # Input layer: 1 input (x) -> n_neurons
        layers.append(nn.Linear(1, n_neurons))
        layers.append(nn.Tanh())
        
        # Hidden layers: n_neurons -> n_neurons
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.Tanh())
        
        # Output layer: n_neurons -> 1 output (u)
        layers.append(nn.Linear(n_neurons, 1))
        
        # Combine all layers into a sequential model
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        """
        Forward pass: compute u(x) from input x.
        
        Args:
            x: Input tensor of shape (N, 1)
        Returns:
            u: Output tensor of shape (N, 1)
        """
        return self.network(x)

# Create the model and move to device
model = PINN(n_hidden=3, n_neurons=32).to(device)

# Print model architecture
print("Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


## Step 3: Define the Physics Loss

This is the **core idea** of PINNs. We define a loss function that penalizes
the network for violating the physics equation.

**Total Loss = PDE Loss + Boundary Loss**

1. **PDE Loss**: How well does the network satisfy $\frac{d^2u}{dx^2} + \sin(\pi x) = 0$?
2. **Boundary Loss**: How well does the network satisfy $u(0) = 0$ and $u(1) = 0$?

### How do we compute derivatives?
We use **automatic differentiation** (autograd in PyTorch):
- PyTorch can compute $\frac{du}{dx}$ and $\frac{d^2u}{dx^2}$ automatically!
- This is why we need `x.requires_grad_(True)` — it tells PyTorch to track gradients


In [ ]:
# ============================================================
# STEP 3: Physics Loss Function
# ============================================================

def compute_pde_residual(model, x):
    """
    Compute the PDE residual: d²u/dx² + sin(πx) = 0
    
    We use torch.autograd.grad to compute derivatives:
    - First call:  du/dx  (first derivative)
    - Second call: d²u/dx² (second derivative)
    
    Args:
        model: The neural network
        x: Collocation points where we enforce the PDE, shape (N, 1)
    
    Returns:
        residual: PDE residual at each point, shape (N, 1)
    """
    # Enable gradient tracking for input x
    x.requires_grad_(True)
    
    # Forward pass: compute u(x)
    u = model(x)
    
    # First derivative: du/dx
    # - grad_outputs=torch.ones_like(u): we want gradient of each output
    # - create_graph=True: we need to differentiate again for 2nd derivative
    du_dx = torch.autograd.grad(
        outputs=u,
        inputs=x,
        grad_outputs=torch.ones_like(u),
        create_graph=True,    # Keep the computation graph for 2nd derivative
        retain_graph=True     # Don't free the graph yet
    )[0]  # [0] because grad returns a tuple
    
    # Second derivative: d²u/dx²
    d2u_dx2 = torch.autograd.grad(
        outputs=du_dx,
        inputs=x,
        grad_outputs=torch.ones_like(du_dx),
        create_graph=True
    )[0]
    
    # PDE residual: d²u/dx² + sin(πx) should = 0
    # (if the network perfectly satisfies the PDE, residual = 0)
    source_term = -torch.sin(np.pi * x)
    residual = d2u_dx2 - source_term
    
    return residual


def compute_loss(model, x_pde, x_bc, u_bc):
    """
    Compute total PINN loss = PDE loss + Boundary Condition loss.
    
    Args:
        model: The neural network
        x_pde: Collocation points for PDE (interior points), shape (N_pde, 1)
        x_bc: Boundary points, shape (N_bc, 1)  
        u_bc: Known boundary values, shape (N_bc, 1)
    
    Returns:
        total_loss, pde_loss, bc_loss: Individual loss components
    """
    # 1. PDE Loss: How well does the network satisfy the heat equation?
    #    We want the residual to be zero everywhere inside the domain
    pde_residual = compute_pde_residual(model, x_pde)
    pde_loss = torch.mean(pde_residual ** 2)  # Mean Squared Error
    
    # 2. Boundary Condition Loss: Does u match the known values at boundaries?
    u_pred_bc = model(x_bc)
    bc_loss = torch.mean((u_pred_bc - u_bc) ** 2)
    
    # 3. Total loss = PDE loss + weighted BC loss
    # We weight BC loss higher to ensure boundaries are satisfied first
    w_bc = 10.0  # Boundary condition weight (higher = stricter enforcement)
    total_loss = pde_loss + w_bc * bc_loss
    
    return total_loss, pde_loss, bc_loss

print("Loss functions defined successfully!")


## Step 4: Prepare Training Data

For PINNs, we need two types of points:

1. **Collocation points** (interior): Random points inside the domain [0, 1]
   where we enforce the PDE. No labels needed — the physics IS the label!

2. **Boundary points**: Points at x=0 and x=1 where we know the solution.

**Key insight**: Unlike traditional ML, we don't need measured data!
The physics equation itself provides the supervision signal.


In [ ]:
# ============================================================
# STEP 4: Prepare Training Points
# ============================================================

# Number of points
N_pde = 100   # Collocation points inside the domain
N_bc = 2      # Boundary points (just x=0 and x=1)

# Collocation points: randomly sampled in (0, 1)
# These are where we enforce the PDE
x_pde = torch.rand(N_pde, 1, device=device)  # Uniform random in [0, 1]

# Boundary condition points and their known values
# At x=0: u=0,  At x=1: u=0
x_bc = torch.tensor([[0.0], [1.0]], device=device)
u_bc = torch.tensor([[0.0], [0.0]], device=device)

# Exact solution for comparison (for plotting only, NOT used in training!)
x_exact = torch.linspace(0, 1, 200).reshape(-1, 1).to(device)
u_exact = torch.sin(np.pi * x_exact) / (np.pi ** 2)

print(f"PDE collocation points: {x_pde.shape}")
print(f"Boundary points:        {x_bc.shape}")
print(f"\nNote: We use {N_pde} interior points + {N_bc} boundary points")
print("No measured data needed — physics provides the supervision!")


## Step 5: Train the PINN

Training loop:
1. Compute the total loss (PDE + boundary conditions)
2. Backpropagate to compute gradients
3. Update network weights
4. Repeat until convergence

We use the **Adam optimizer** with a learning rate scheduler that reduces
the learning rate when the loss plateaus.


In [ ]:
# ============================================================
# STEP 5: Training Loop
# ============================================================

# Optimizer: Adam is a good default for PINNs
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Learning rate scheduler: reduce LR by 0.5 when loss plateaus for 500 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=500, factor=0.5, min_lr=1e-6, verbose=True
)

# Training parameters
n_epochs = 5000
print_every = 500

# Storage for loss history
history = {'total': [], 'pde': [], 'bc': []}

print("Starting PINN training...")
print("=" * 60)

for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Resample collocation points each epoch for better coverage
    # This is a common trick in PINN training
    x_pde_train = torch.rand(N_pde, 1, device=device, requires_grad=True)
    
    # Compute loss
    total_loss, pde_loss, bc_loss = compute_loss(model, x_pde_train, x_bc, u_bc)
    
    # Backpropagation: compute gradients of loss w.r.t. network weights
    total_loss.backward()
    
    # Update weights
    optimizer.step()
    
    # Update learning rate
    scheduler.step(total_loss)
    
    # Record losses
    history['total'].append(total_loss.item())
    history['pde'].append(pde_loss.item())
    history['bc'].append(bc_loss.item())
    
    # Print progress
    if epoch % print_every == 0 or epoch == 1:
        print(f"Epoch {epoch:5d}/{n_epochs} | "
              f"Total: {total_loss.item():.2e} | "
              f"PDE: {pde_loss.item():.2e} | "
              f"BC: {bc_loss.item():.2e}")

print("=" * 60)
print("Training complete!")


## Step 6: Visualize Results

Let's compare the PINN prediction with the exact analytical solution
and examine the training convergence.


In [ ]:
# ============================================================
# STEP 6: Plot Results
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: PINN Prediction vs Exact Solution ---
model.eval()
with torch.no_grad():
    u_pred = model(x_exact).cpu().numpy()

x_np = x_exact.cpu().numpy()
u_ex_np = u_exact.cpu().numpy()

axes[0].plot(x_np, u_ex_np, 'b-', linewidth=2, label='Exact Solution')
axes[0].plot(x_np, u_pred, 'r--', linewidth=2, label='PINN Prediction')
axes[0].scatter([0, 1], [0, 0], c='green', s=100, zorder=5, label='Boundary Conditions')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('u(x)', fontsize=12)
axes[0].set_title('PINN vs Exact Solution', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Absolute Error ---
error = np.abs(u_pred - u_ex_np)
axes[1].plot(x_np, error, 'k-', linewidth=2)
axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('|u_pred - u_exact|', fontsize=12)
axes[1].set_title(f'Absolute Error (max = {error.max():.2e})', fontsize=14)
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

# --- Plot 3: Training Loss History ---
axes[2].semilogy(history['total'], label='Total Loss', alpha=0.8)
axes[2].semilogy(history['pde'], label='PDE Loss', alpha=0.8)
axes[2].semilogy(history['bc'], label='BC Loss', alpha=0.8)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Loss', fontsize=12)
axes[2].set_title('Training Loss History', fontsize=14)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final accuracy
print(f"\nMax absolute error: {error.max():.6e}")
print(f"Mean absolute error: {error.mean():.6e}")
print(f"Relative L2 error: {np.linalg.norm(u_pred - u_ex_np) / np.linalg.norm(u_ex_np):.6e}")


## Key Takeaways

1. **No training data needed**: The PINN learned the solution purely from the physics equation + boundary conditions.

2. **Three loss components**:
   - PDE loss: enforces $d^2u/dx^2 = -\sin(\pi x)$ at collocation points
   - BC loss: enforces $u(0) = u(1) = 0$
   - Total loss = PDE loss + w × BC loss

3. **Automatic differentiation** is the key technology: PyTorch computes $du/dx$ and $d^2u/dx^2$ for us.

4. **Common tricks**:
   - Use `tanh` activation (smooth, infinitely differentiable)
   - Resample collocation points each epoch
   - Weight the BC loss higher than PDE loss
   - Use learning rate scheduling

### Next Steps
- Try changing the number of collocation points (N_pde)
- Try different network sizes (n_hidden, n_neurons)
- Try different source terms in the PDE
- Move on to **Project 2: Time-dependent Burgers' equation**
